In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from config import SIM_CONFIG
from users import generate_users
from sessions import generate_sessions
from events import generate_events
from orders import generate_orders
from order_items import generate_order_items

rng = np.random.default_rng(seed=42)
print("modules loaded successfully")

modules loaded successfully


In [2]:
master_ss = np.random.SeedSequence(SIM_CONFIG["random_seed"])
stage_seeds = master_ss.spawn(5)
rng_users = np.random.default_rng(stage_seeds[0])
rng_sessions = np.random.default_rng(stage_seeds[1])
rng_events = np.random.default_rng(stage_seeds[2])
rng_orders = np.random.default_rng(stage_seeds[3])
rng_order_items = np.random.default_rng(stage_seeds[4])

n_workers = SIM_CONFIG.get("n_workers", 1)

# test_users_df = generate_users(SIM_CONFIG["target_users"], rng_users)
df_test_users = generate_users(200000, rng_users)
df_test_sessions = generate_sessions(df_test_users, rng_sessions)
df_test_events = generate_events(df_test_sessions, df_test_users, rng_events, n_workers)
df_test_orders = generate_orders(df_test_events, df_test_sessions, rng_orders)
df_test_order_items = generate_order_items(df_test_orders, df_test_users, rng_order_items)
print("all tables generated successfully in memory")

print("\nEXPORTING TO CSV")
current_script_dir = os.path.abspath(os.getcwd())
output_dir = os.path.abspath(os.path.join(current_script_dir, '..', 'test_data'))
os.makedirs(output_dir, exist_ok=True)

df_clean_users = df_test_users.drop(columns=[
    "latent_income_score",
    "latent_digital_literacy",
    "latent_trust_in_platform",
])

df_sorted_clean_users = df_clean_users.sort_values(by="account_created_at", ascending=True).reset_index(drop=True)
df_sorted_sessions = df_test_sessions.sort_values(by="session_start_time", ascending=True).reset_index(drop=True)
df_sorted_events = df_test_events.sort_values(by="event_timestamp").reset_index(drop=True)
df_sorted_orders = df_test_orders.sort_values(by="order_timestamp").reset_index(drop=True)

df_sorted_clean_users.to_csv(os.path.join(output_dir, 'users.csv'), index=False)
df_sorted_sessions.to_csv(os.path.join(output_dir, 'sessions.csv'), index=False)
df_sorted_events.to_csv(os.path.join(output_dir, 'events.csv'), index=False)
df_sorted_orders.to_csv(os.path.join(output_dir, 'orders.csv'), index=False)
df_test_order_items.to_csv(os.path.join(output_dir, 'order_items.csv'), index=False)

print(f"export completed in {output_dir}")

Generating 200000 users
users table generated!
Generating 1661641 sessions
sessions table generated!
Generating events for 1661641 sessions
Generating 51511 orders for purchases events
orders table generated!
Generating order items for 51511 orders
order items table generated!
all tables generated successfully in memory

EXPORTING TO CSV
export completed in /home/ruicchi/github-projects/ph-ecommerce-data-simulator/test_data
